# Ensemble Methods

In [1]:
import numpy as np
import pandas as pd
from io import StringIO
from IPython.display import display
from sklearn.base import clone
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.ensemble import VotingRegressor, StackingRegressor, VotingClassifier

pd.set_option("display.precision", 4)
SEED = 42

## Choose your own Regression

**Working together with your partner**, choose an interesting data set to fit *regression* models for, that is, a data set with a *quantitative* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *regression* models to predict the *quantitative* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a linear regression model
- One partner should fit a k-nearest neighbors regression model

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create two more models**:

- An ensemble model that uses **voting**
- An ensemble model that uses **stacking**

For each of the four models (linear regression, knn regression, voting ensemble, stacking ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test RMSE using cross-validation, and compare the results

Which model seems to work best?

I chose the diabetes dataset included with scikit-learn. It has 442 patients and 10 predictors, including age, BMI, blood pressure, and blood measurements. The quantitative target measures disease progression one year after baseline.

I reserve 20% of the data for example predictions. I estimate test RMSE with five-fold cross-validation on the remaining 80%, using the same folds for every model. Scaling is inside each pipeline, so it is learned only from the training portion of each fold.

In [2]:
diabetes = load_diabetes(as_frame=True, scaled=False)
X_reg = diabetes.data
y_reg = diabetes.target
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.20, random_state=SEED
)
reg_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Regression: {len(X_reg_train)} training patients, {len(X_reg_test)} reserved patients")
display(X_reg_train.head())

Regression: 353 training patients, 89 reserved patients


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
17,68.0,2.0,27.5,111.0,214.0,147.0,39.0,5.0,4.9416,91.0
66,46.0,2.0,24.7,85.0,174.0,123.2,30.0,6.0,4.6444,96.0
137,50.0,1.0,31.0,123.0,178.0,105.0,48.0,4.0,4.8283,88.0
245,41.0,1.0,23.1,86.0,148.0,78.0,58.0,3.0,4.0943,60.0
31,42.0,1.0,20.3,71.0,161.0,81.2,66.0,2.0,4.2341,81.0


In [3]:
linear = Pipeline([
    ("scale", StandardScaler()),
    ("model", LinearRegression())
])
knn_reg = Pipeline([
    ("scale", StandardScaler()),
    ("model", KNeighborsRegressor(n_neighbors=10))
])

# Voting gives the two predictions equal weight.
voting_reg = VotingRegressor(estimators=[
    ("linear", clone(linear)), ("knn", clone(knn_reg))
])

# Stacking trains Ridge on inner out-of-fold predictions from the base models.
stacking_reg = StackingRegressor(
    estimators=[("linear", clone(linear)), ("knn", clone(knn_reg))],
    final_estimator=Ridge(alpha=1.0),
    cv=KFold(n_splits=5, shuffle=True, random_state=SEED)
)
reg_models = {
    "Linear regression": linear,
    "KNN regression (k=10)": knn_reg,
    "Voting regression": voting_reg,
    "Stacking regression": stacking_reg
}

reg_rows = []
for name, model in reg_models.items():
    rmse = -cross_val_score(
        model, X_reg_train, y_reg_train, cv=reg_cv,
        scoring="neg_root_mean_squared_error"
    )
    reg_rows.append({"Model": name, "Mean CV test RMSE": rmse.mean(),
                     "Fold SD": rmse.std(ddof=1)})
    model.fit(X_reg_train, y_reg_train)

reg_results = pd.DataFrame(reg_rows).set_index("Model").sort_values("Mean CV test RMSE")
print("REGRESSION ANSWER: Lower RMSE is better")
display(reg_results)

REGRESSION ANSWER: Lower RMSE is better


,Mean CV test RMSE,Fold SD
Model,,
Linear regression,55.3946,2.6402
Stacking regression,55.4767,2.8209
Voting regression,56.2312,3.2768
KNN regression (k=10),59.5327,3.8481


In [4]:
reg_examples = X_reg_test.iloc[:5]
reg_predictions = pd.DataFrame({"Actual progression": y_reg_test.loc[reg_examples.index]})
for name, model in reg_models.items():
    reg_predictions[name] = model.predict(reg_examples)
reg_predictions.index.name = "Patient index"
print("REGRESSION ANSWER: Predictions for five reserved patients")
display(reg_predictions.round(2))
print("Best regression model by mean CV test RMSE:", reg_results.index[0])

REGRESSION ANSWER: Predictions for five reserved patients


,Actual progression,Linear regression,KNN regression (k=10),Voting regression,Stacking regression
Patient index,,,,,
287,219.0,139.55,148.7,144.12,140.80
211,70.0,179.52,160.1,169.81,178.30
72,202.0,134.04,157.3,145.67,136.17
321,230.0,291.42,240.6,266.01,285.86
73,111.0,123.79,158.6,141.19,126.78


Best regression model by mean CV test RMSE: Linear regression


**Regression answer:** Linear regression works best in this comparison, with the lowest mean cross-validation test RMSE of **55.39**. Linear regression has an RMSE of 55.39, KNN has 59.53, voting has 56.23, and stacking has 55.48. The five example predictions show that the models can give different estimates for the same patient. Voting averages the two base predictions, while stacking learns how to combine them using inner cross-validation predictions. An ensemble is not guaranteed to improve on the individual models. These are estimates from this split and these settings; the fold SD shows how much results vary across the five folds.

## Voting in Classification

Suppose you have three models for predicting a categorical target:

- k-nearest neighbors
- logistic regression
- decision tree

How might you apply voting to create an ensemble model? Discuss conceptually with your partner. Create a concrete example to illustrate your method. Hint: There are two basic approaches to voting in classification; can you think of both?

Note: You can assume a binary target, but the general idea of voting applies to multiclass problems. However, if any of the candidate *models* are not naturally multiclass (like logistic regression) then you have to consider things like OvR and OvO as we have discussed.

**Hard voting:** Each model predicts a class, and the class with the most votes wins.

**Soft voting:** Average the models' predicted probabilities for each class, then choose the class with the highest average. For a binary target with equal weights, predict class 1 when its average probability is greater than 0.5. Soft voting uses confidence as well as the predicted class; the probability estimates should be reasonably calibrated.

For this example, the predictions disagree:

| Model | P(class 1) | Predicted class |
| --- | ---: | ---: |
| KNN | 0.51 | 1 |
| Logistic regression | 0.51 | 1 |
| Decision tree | 0.01 | 0 |

Hard voting predicts **1** because it gets two of the three votes. Soft voting predicts **0** because the average probability of class 1 is (0.51 + 0.51 + 0.01) / 3 = **0.3433**. These are illustrative probabilities, not fitted results. For multiclass targets, hard voting still counts class votes, and soft voting averages each class's probabilities and selects the largest.

In [5]:
example_probabilities = np.array([0.51, 0.51, 0.01])
example_classes = (example_probabilities > 0.5).astype(int)
hard_vote = np.bincount(example_classes, minlength=2).argmax()
soft_probability = example_probabilities.mean()
soft_vote = int(soft_probability > 0.5)
print(f"Hard voting prediction: {hard_vote}")
print(f"Soft voting average P(class 1): {soft_probability:.4f}")
print(f"Soft voting prediction: {soft_vote}")

Hard voting prediction: 1
Soft voting average P(class 1): 0.3433
Soft voting prediction: 0


## Choose your own Classification

**Working together with your partner**, choose an interesting data set to fit *classification* models for, that is, a data set with a *categorical* target variable and several feature variables. Could be a data set that: we have used in class, we have used on an assignment, a data set from your project, you used in another course, or you just find interesting! You'll work together with your partner to develop *classification* models to predict the *categorical* target (so you need to decide on the data set and target variable together with your partner).

After you have decided on the data set and target variable:

- One partner should fit a k-nearest neighbors classification model
- One partner should fit another classification model like decision tree or logistic regression

If you're using an example from earlier in the class, you can use a model you have already come up with or you can try something new. Note: you do NOT have to agree on the features included in your models.

After each partner has their model, **work together to combine your code and create an ensemble model that uses *voting***.

For each of the three models (knn classification, other classification, voting ensemble)

1. Fit the model using the training data
1. Use the fitted model to predict the target for a few observations of your choice, and compare the predictions
1. Estimate the test macro precision, recall, and F1 score using cross-validation, and compare the results

Which model seems to work best?

**Hint:** [`VotingClassifier`](https://scikit-learn.org/1.5/modules/generated/sklearn.ensemble.VotingClassifier.html) works almost identically to `VotingRegressor`.

I chose the heart-attack dataset from the earlier activity. The target is `output`: 0 means not at risk and 1 means at risk according to the recorded diagnosis. I use age, resting blood pressure, cholesterol, maximum heart rate, sex, chest-pain type, and resting ECG as predictors.

I reserve 20% of patients for example predictions, stratifying by the target. On the training data, I use the same five stratified folds for all three models. Numeric predictors are scaled and categorical predictors are one-hot encoded inside each pipeline. I compare the models using macro F1, with macro precision and recall also reported. Macro averages give each class equal weight.

In [6]:
# Course data embedded so the notebook runs without a download.
heart = pd.read_csv(StringIO('age,sex,cp,trtbps,chol,restecg,thalach,output\n63,1,3,145,233,0,150,1\n37,1,2,130,250,1,187,1\n56,1,1,120,236,1,178,1\n57,0,0,120,354,1,163,1\n57,1,0,140,192,1,148,1\n56,0,1,140,294,0,153,1\n44,1,1,120,263,1,173,1\n52,1,2,172,199,1,162,1\n57,1,2,150,168,1,174,1\n54,1,0,140,239,1,160,1\n48,0,2,130,275,1,139,1\n49,1,1,130,266,1,171,1\n58,0,3,150,283,0,162,1\n50,0,2,120,219,1,158,1\n58,0,2,120,340,1,172,1\n66,0,3,150,226,1,114,1\n43,1,0,150,247,1,171,1\n69,0,3,140,239,1,151,1\n44,1,2,130,233,1,179,1\n61,1,2,150,243,1,137,1\n71,0,1,160,302,1,162,1\n59,1,2,150,212,1,157,1\n51,1,2,110,175,1,123,1\n65,0,2,140,417,0,157,1\n53,1,2,130,197,0,152,1\n41,0,1,105,198,1,168,1\n65,1,0,120,177,1,140,1\n44,1,1,130,219,0,188,1\n51,1,3,125,213,0,125,1\n46,0,2,142,177,0,160,1\n54,0,2,135,304,1,170,1\n54,1,2,150,232,0,165,1\n65,0,2,155,269,1,148,1\n65,0,2,160,360,0,151,1\n51,0,2,140,308,0,142,1\n48,1,1,130,245,0,180,1\n45,1,0,104,208,0,148,1\n53,0,0,130,264,0,143,1\n39,1,2,140,321,0,182,1\n52,1,1,120,325,1,172,1\n44,1,2,140,235,0,180,1\n47,1,2,138,257,0,156,1\n53,0,2,128,216,0,115,1\n53,0,0,138,234,0,160,1\n51,0,2,130,256,0,149,1\n66,1,0,120,302,0,151,1\n62,1,2,130,231,1,146,1\n44,0,2,108,141,1,175,1\n63,0,2,135,252,0,172,1\n48,1,0,122,222,0,186,1\n45,1,0,115,260,0,185,1\n34,1,3,118,182,0,174,1\n57,0,0,128,303,0,159,1\n54,1,1,108,309,1,156,1\n52,1,3,118,186,0,190,1\n41,1,1,135,203,1,132,1\n58,1,2,140,211,0,165,1\n35,0,0,138,183,1,182,1\n51,1,2,100,222,1,143,1\n45,0,1,130,234,0,175,1\n44,1,1,120,220,1,170,1\n62,0,0,124,209,1,163,1\n54,1,2,120,258,0,147,1\n51,1,2,94,227,1,154,1\n29,1,1,130,204,0,202,1\n51,1,0,140,261,0,186,1\n43,0,2,122,213,1,165,1\n51,1,2,125,245,0,166,1\n59,1,1,140,221,1,164,1\n52,1,1,128,205,1,184,1\n41,1,2,112,250,1,179,1\n45,1,1,128,308,0,170,1\n60,0,2,102,318,1,160,1\n52,1,3,152,298,1,178,1\n42,0,0,102,265,0,122,1\n67,0,2,115,564,0,160,1\n54,0,2,110,214,1,158,1\n58,0,0,100,248,0,122,1\n57,1,0,132,207,1,168,1\n52,1,2,138,223,1,169,1\n54,0,1,132,288,0,159,1\n45,0,1,112,160,1,138,1\n53,1,0,142,226,0,111,1\n62,0,0,140,394,0,157,1\n52,1,0,108,233,1,147,1\n43,1,2,130,315,1,162,1\n42,1,3,148,244,0,178,1\n59,1,3,178,270,0,145,1\n63,0,1,140,195,1,179,1\n42,1,2,120,240,1,194,1\n50,1,2,129,196,1,163,1\n68,0,2,120,211,0,115,1\n45,0,0,138,236,0,152,1\n64,0,0,180,325,1,154,1\n57,1,2,150,126,1,173,1\n64,0,2,140,313,1,133,1\n43,1,0,110,211,1,161,1\n55,1,1,130,262,1,155,1\n37,0,2,120,215,1,170,1\n41,1,2,130,214,0,168,1\n56,1,3,120,193,0,162,1\n46,0,1,105,204,1,172,1\n46,0,0,138,243,0,152,1\n64,0,0,130,303,1,122,1\n41,0,2,112,268,0,172,1\n54,0,2,108,267,0,167,1\n39,0,2,94,199,1,179,1\n34,0,1,118,210,1,192,1\n67,0,2,152,277,1,172,1\n52,0,2,136,196,0,169,1\n74,0,1,120,269,0,121,1\n54,0,2,160,201,1,163,1\n49,0,1,134,271,1,162,1\n42,1,1,120,295,1,162,1\n41,1,1,110,235,1,153,1\n41,0,1,126,306,1,163,1\n49,0,0,130,269,1,163,1\n60,0,2,120,178,1,96,1\n62,1,1,128,208,0,140,1\n57,1,0,110,201,1,126,1\n64,1,0,128,263,1,105,1\n51,0,2,120,295,0,157,1\n43,1,0,115,303,1,181,1\n42,0,2,120,209,1,173,1\n67,0,0,106,223,1,142,1\n76,0,2,140,197,2,116,1\n70,1,1,156,245,0,143,1\n44,0,2,118,242,1,149,1\n60,0,3,150,240,1,171,1\n44,1,2,120,226,1,169,1\n42,1,2,130,180,1,150,1\n66,1,0,160,228,0,138,1\n71,0,0,112,149,1,125,1\n64,1,3,170,227,0,155,1\n66,0,2,146,278,0,152,1\n39,0,2,138,220,1,152,1\n58,0,0,130,197,1,131,1\n47,1,2,130,253,1,179,1\n35,1,1,122,192,1,174,1\n58,1,1,125,220,1,144,1\n56,1,1,130,221,0,163,1\n56,1,1,120,240,1,169,1\n55,0,1,132,342,1,166,1\n41,1,1,120,157,1,182,1\n38,1,2,138,175,1,173,1\n38,1,2,138,175,1,173,1\n67,1,0,160,286,0,108,0\n67,1,0,120,229,0,129,0\n62,0,0,140,268,0,160,0\n63,1,0,130,254,0,147,0\n53,1,0,140,203,0,155,0\n56,1,2,130,256,0,142,0\n48,1,1,110,229,1,168,0\n58,1,2,132,224,0,173,0\n60,1,0,130,206,0,132,0\n40,1,0,110,167,0,114,0\n60,1,0,117,230,1,160,0\n64,1,2,140,335,1,158,0\n43,1,0,120,177,0,120,0\n57,1,0,150,276,0,112,0\n55,1,0,132,353,1,132,0\n65,0,0,150,225,0,114,0\n58,1,2,112,230,0,165,0\n50,1,0,150,243,0,128,0\n44,1,0,112,290,0,153,0\n60,1,0,130,253,1,144,0\n54,1,0,124,266,0,109,0\n50,1,2,140,233,1,163,0\n41,1,0,110,172,0,158,0\n51,0,0,130,305,1,142,0\n58,1,0,128,216,0,131,0\n54,1,0,120,188,1,113,0\n60,1,0,145,282,0,142,0\n60,1,2,140,185,0,155,0\n59,1,0,170,326,0,140,0\n46,1,2,150,231,1,147,0\n67,1,0,125,254,1,163,0\n62,1,0,120,267,1,99,0\n65,1,0,110,248,0,158,0\n44,1,0,110,197,0,177,0\n60,1,0,125,258,0,141,0\n58,1,0,150,270,0,111,0\n68,1,2,180,274,0,150,0\n52,1,0,128,255,1,161,0\n59,1,0,110,239,0,142,0\n60,0,0,150,258,0,157,0\n49,1,2,120,188,1,139,0\n57,1,2,128,229,0,150,0\n61,1,0,120,260,1,140,0\n39,1,0,118,219,1,140,0\n61,0,0,145,307,0,146,0\n56,1,0,125,249,0,144,0\n43,0,0,132,341,0,136,0\n62,0,2,130,263,1,97,0\n63,1,0,130,330,0,132,0\n65,1,0,135,254,0,127,0\n63,0,0,150,407,0,154,0\n65,1,3,138,282,0,174,0\n56,0,0,200,288,0,133,0\n54,1,0,110,239,1,126,0\n70,1,0,145,174,1,125,0\n62,1,1,120,281,0,103,0\n35,1,0,120,198,1,130,0\n59,1,3,170,288,0,159,0\n64,1,2,125,309,1,131,0\n47,1,2,108,243,1,152,0\n57,1,0,165,289,0,124,0\n55,1,0,160,289,0,145,0\n64,1,0,120,246,0,96,0\n70,1,0,130,322,0,109,0\n51,1,0,140,299,1,173,0\n58,1,0,125,300,0,171,0\n60,1,0,140,293,0,170,0\n77,1,0,125,304,0,162,0\n35,1,0,126,282,0,156,0\n70,1,2,160,269,1,112,0\n59,0,0,174,249,1,143,0\n64,1,0,145,212,0,132,0\n57,1,0,152,274,1,88,0\n56,1,0,132,184,0,105,0\n48,1,0,124,274,0,166,0\n56,0,0,134,409,0,150,0\n66,1,1,160,246,1,120,0\n54,1,1,192,283,0,195,0\n69,1,2,140,254,0,146,0\n43,1,0,132,247,0,143,0\n62,0,0,138,294,1,106,0\n59,1,3,160,273,0,125,0\n58,1,0,128,259,0,130,0\n50,1,0,144,200,0,126,0\n62,0,0,150,244,1,154,0\n38,1,3,120,231,1,182,0\n66,0,0,178,228,1,165,0\n52,1,0,112,230,1,160,0\n53,1,0,123,282,1,95,0\n63,0,0,108,269,1,169,0\n54,1,0,110,206,0,108,0\n66,1,0,112,212,0,132,0\n55,0,0,180,327,2,117,0\n49,1,2,118,149,0,126,0\n54,1,0,122,286,0,116,0\n56,1,0,130,283,0,103,0\n46,1,0,120,249,0,144,0\n61,1,3,134,234,1,145,0\n67,1,0,120,237,1,71,0\n58,1,0,100,234,1,156,0\n52,1,0,125,212,1,168,0\n58,1,0,146,218,1,105,0\n57,1,1,124,261,1,141,0\n58,0,1,136,319,0,152,0\n61,1,0,138,166,0,125,0\n42,1,0,136,315,1,125,0\n52,1,0,128,204,1,156,0\n59,1,2,126,218,1,134,0\n40,1,0,152,223,1,181,0\n61,1,0,140,207,0,138,0\n46,1,0,140,311,1,120,0\n59,1,3,134,204,1,162,0\n57,1,1,154,232,0,164,0\n57,1,0,110,335,1,143,0\n55,0,0,128,205,2,130,0\n61,1,0,148,203,1,161,0\n58,1,0,114,318,2,140,0\n58,0,0,170,225,0,146,0\n67,1,2,152,212,0,150,0\n44,1,0,120,169,1,144,0\n63,1,0,140,187,0,144,0\n63,0,0,124,197,1,136,0\n59,1,0,164,176,0,90,0\n57,0,0,140,241,1,123,0\n45,1,3,110,264,1,132,0\n68,1,0,144,193,1,141,0\n57,0,1,130,236,0,174,0\n'))

numeric_features = ["age", "trtbps", "chol", "thalach"]
categorical_features = ["sex", "cp", "restecg"]
X_clf = heart[numeric_features + categorical_features]
y_clf = heart["output"]
X_clf_train, X_clf_test, y_clf_train, y_clf_test = train_test_split(
    X_clf, y_clf, test_size=0.20, random_state=SEED, stratify=y_clf
)
clf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(f"Classification: {len(X_clf_train)} training patients, {len(X_clf_test)} reserved patients")
print("Target counts in the full dataset:")
display(y_clf.value_counts().sort_index().rename("Patients").to_frame())

preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])
knn_clf = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", KNeighborsClassifier(n_neighbors=10))
])
logistic_clf = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(max_iter=2000, random_state=SEED))
])
voting_clf = VotingClassifier(
    estimators=[("knn", clone(knn_clf)), ("logistic", clone(logistic_clf))],
    voting="soft"
)
clf_models = {
    "KNN classification (k=10)": knn_clf,
    "Logistic regression": logistic_clf,
    "Soft voting classification": voting_clf
}

Classification: 218 training patients, 55 reserved patients
Target counts in the full dataset:


,Patients
output,
0,127
1,146


In [7]:
scoring = {"Macro precision": "precision_macro", "Macro recall": "recall_macro", "Macro F1": "f1_macro"}
clf_rows = []
for name, model in clf_models.items():
    scores = cross_validate(model, X_clf_train, y_clf_train, cv=clf_cv, scoring=scoring)
    row = {"Model": name}
    for metric in scoring:
        row[metric] = scores[f"test_{metric}"].mean()
    row["F1 fold SD"] = scores["test_Macro F1"].std(ddof=1)
    clf_rows.append(row)
    model.fit(X_clf_train, y_clf_train)

clf_results = pd.DataFrame(clf_rows).set_index("Model").sort_values("Macro F1", ascending=False)
print("CLASSIFICATION ANSWER: Mean test-fold macro scores; higher is better")
display(clf_results)

clf_examples = X_clf_test.iloc[:5]
clf_predictions = pd.DataFrame({"Actual class": y_clf_test.loc[clf_examples.index]})
for name, model in clf_models.items():
    clf_predictions[name] = model.predict(clf_examples)
clf_predictions.index.name = "Patient index"
print("CLASSIFICATION ANSWER: Predictions for five reserved patients")
display(clf_predictions)

example_probs = pd.DataFrame(index=clf_examples.index)
for name, model in clf_models.items():
    positive_column = list(model.classes_).index(1)
    example_probs[f"{name}: P(class 1)"] = model.predict_proba(clf_examples)[:, positive_column]
print("Class 1 probabilities used to compare the soft-voting predictions:")
display(example_probs)
print("Best classification model by mean CV macro F1:", clf_results.index[0])

CLASSIFICATION ANSWER: Mean test-fold macro scores; higher is better


,Macro precision,Macro recall,Macro F1,F1 fold SD
Model,,,,
Logistic regression,0.7945,0.7912,0.7917,0.0772
Soft voting classification,0.7714,0.7700,0.7694,0.0683
KNN classification (k=10),0.7550,0.7555,0.7519,0.0511


CLASSIFICATION ANSWER: Predictions for five reserved patients


,Actual class,KNN classification (k=10),Logistic regression,Soft voting classification
Patient index,,,,
252,0,0,0,0
147,0,0,0,0
89,1,1,1,1
217,0,0,0,0
248,0,1,1,1


Class 1 probabilities used to compare the soft-voting predictions:


,KNN classification (k=10): P(class 1),Logistic regression: P(class 1),Soft voting classification: P(class 1)
252,0.5,0.3175,0.4087
147,0.0,0.0721,0.0361
89,0.9,0.9385,0.9193
217,0.2,0.0675,0.1337
248,0.8,0.6867,0.7433


Best classification model by mean CV macro F1: Logistic regression


**Classification answer:** Logistic regression works best by mean cross-validation macro F1: **0.7917**. Its macro precision is 0.7945 and macro recall is 0.7912. KNN has macro F1 of 0.7519, logistic regression has 0.7917, and soft voting has 0.7694. All three models agree on the five example classifications. They correctly classify four patients and incorrectly predict class 1 for patient 248, whose actual class is 0. Soft voting averages the two models' class probabilities with equal weights. It can change the prediction when one model is more confident, even when the two predicted classes disagree. With two models, hard voting can tie, which is one reason I chose soft voting here. The differences are small, so this result does not establish that the winning model will always perform better.